In [1]:
from sunpy.coordinates.sun import carrington_rotation_time
from astropy.io import fits
import astropy.units as u
from astropy.time import Time
import numpy as np
import sunpy.map
import drms
import subprocess
import sys, os, getopt

# NRT Ingestion

In [2]:
#record_set_query = 'hmi.m_720s_nrt[2022.03.21_23:12:00_TAI-2022.03.26_12:00:00_TAI]'
record_set_query = 'hmi.m_720s_nrt[2022.03.26_12:00:00_TAI-2022.03.28_12:00:00_TAI]'
path = '../output/data/nrt/cr2255/'

In [3]:
client = drms.Client(server='JSOC', email='loeschl@mps.mpg.de')
#record_set_query = 'hmi.m_720s_nrt[2022.03.25_00:00/24m]'
export_request = client.export(record_set_query, method='url', protocol='fits')

In [4]:
export_request.wait()

export_request.status

export_request.download(path)

DrmsExportError: One or more Storage Units are invalid1499088026 [status=4]

In [6]:
pwd

'/scratch/slam/loeschl/dev/python/synop/sim/los'

In [7]:
files = os.listdir(path)
fitsfiles = [file for file in files if file.endswith(".fits")]

# TODO produce new scripts with the correct series name and ingest/process data to produce new FSM
#data_series_m720s_nrt = "mps_loeschl.phi_m720s"
data_series_m720s_nrt = "mps_loeschl.hmi_m720s_nrt"

setinfo_out = open(path+'set_info.sh', 'w')
setinfo_out.write('#!/bin/bash\n')
#TODO segment names
set_info = 'set_info -c ds="%s" T_REC="%s" magnetogram=%s >> set_info.log 2>&1\n'
set_keys = "set_keys ds=%s[%s] %s=%s"

#trec_out = open(path+'trecs_nrt.txt', 'w')

i = 0
for fname in fitsfiles:
    
    print('Processing %s...' %fname)
    
    # load with scaling to recognize blank cells -> necessary to prevent artifacts after resize
    fld = fits.open(path+fname)#, do_not_scale_image_data=True) 
    trec = fld[1].header['T_REC']
    fld.close()

    setinfo_out.write('\necho %s' %set_info %(data_series_m720s_nrt, trec, fname))
    setinfo_out.write(set_info %(data_series_m720s_nrt, trec, fname))
    #trec_out.write("%s\n"%trec)
    
    i += 1

setinfo_out.write('\necho "done"')
setinfo_out.close()

#trec_out.close()

print('done')

FileNotFoundError: [Errno 2] No such file or directory: '../output/data/nrt/cr2255/'

In [5]:
len(fitsfiles)

NameError: name 'fitsfiles' is not defined

In [6]:
set_info %(data_series_m720s_nrt, trec, fname)

NameError: name 'set_info' is not defined

# Visualisation

In [29]:
%matplotlib widget

In [30]:
t1 = fits.open('../output/data/nrt/hmi.m_720s_nrt.20220325_000000_TAI.3.magnetogram.fits')[1]
t2 = fits.open('/SUM46/D281475007287133/S00000/magnetogram.fits')[1]

In [31]:
fig, (ax1, ax2) = plt.subplots(figsize=(12,6), ncols=2)
ax1.imshow(t1.data, cmap='hmimag', vmin=-1500, vmax=1500, origin="lower")
ax2.imshow(t2.data, cmap='hmimag', vmin=-1500, vmax=1500, origin="lower")

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [32]:
fig, ax1 = plt.subplots(figsize=(12,6))
ax1.imshow(t1.data-t2.data, cmap='hmimag', vmin=-1500, vmax=1500, origin="lower")

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …